In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds

/home/yusupov/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/yusupov/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
import pandas as pd
import gzip

def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def getDF(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

df = getDF('reviews_Sports_and_Outdoors_5.json.gz')

In [3]:
df.head(10)

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AIXZKN4ACSKI,1881509818,David Briner,"[0, 0]",This came in on time and I am veru happy with ...,5.0,Woks very good,1390694400,"01 26, 2014"
1,A1L5P841VIO02V,1881509818,Jason A. Kramer,"[1, 1]",I had a factory Glock tool that I was using fo...,5.0,Works as well as the factory tool,1328140800,"02 2, 2012"
2,AB2W04NI4OEAD,1881509818,J. Fernald,"[2, 2]",If you don't have a 3/32 punch or would like t...,4.0,"It's a punch, that's all.",1330387200,"02 28, 2012"
3,A148SVSWKTJKU6,1881509818,"Jusitn A. Watts ""Maverick9614""","[0, 0]",This works no better than any 3/32 punch you w...,4.0,It's a punch with a Glock logo.,1328400000,"02 5, 2012"
4,AAAWJ6LW9WMOO,1881509818,Material Man,"[0, 0]",I purchased this thinking maybe I need a speci...,4.0,"Ok,tool does what a regular punch does.",1366675200,"04 23, 2013"
5,A2XX2A4OJCDNLZ,1881509818,RatherLiveInKeyWest,"[0, 0]","Needed this tool to really break down my G22, ...",5.0,Glock punch tool - needed for your Glock and o...,1351814400,"11 2, 2012"
6,A283UOBQRUNM4Q,1881509818,Thomas Dragon,"[0, 0]",If u don't have it .. Get it. All you need to ...,5.0,Great tool,1402358400,"06 10, 2014"
7,AWG3H90WVZ0Z1,2094869245,Alec Nelson,"[0, 0]",This light will no doubt capture the attention...,4.0,Bright!,1377907200,"08 31, 2013"
8,A3V52OTJHKIJZX,2094869245,"A. Saenz Jr. ""Bettering self""","[0, 1]","Light and laser torch work well, very bright. ...",5.0,Be seen,1369612800,"05 27, 2013"
9,A3SZBE5F3UQ9EC,2094869245,"ChasRat ""ChasRat""","[0, 0]",Does everything it says it will do. I would li...,5.0,Bicycle rear tail light,1383350400,"11 2, 2013"


In [4]:
new_df = df[['reviewerID', 'asin', 'overall', 'unixReviewTime']].copy()
new_df.columns = ['user_id', 'item_id', 'rating', 'timestamp']
new_df['rating'] = 1
new_df.head(5)

,user_id,item_id,rating,timestamp
0,AIXZKN4ACSKI,1881509818,1,1390694400
1,A1L5P841VIO02V,1881509818,1,1328140800
2,AB2W04NI4OEAD,1881509818,1,1330387200
3,A148SVSWKTJKU6,1881509818,1,1328400000
4,AAAWJ6LW9WMOO,1881509818,1,1366675200


In [5]:
new_df['user_id'], unique_user_ids = pd.factorize(new_df['user_id'])

new_df['item_id'], unique_item_ids = pd.factorize(new_df['item_id'])
new_df['user_id'] += 1
new_df['item_id'] += 1
new_df.head(5)

,user_id,item_id,rating,timestamp
0,1,1,1,1390694400
1,2,1,1,1328140800
2,3,1,1,1330387200
3,4,1,1,1328400000
4,5,1,1,1366675200


In [6]:
df_sorted = new_df.sort_values(by='timestamp')

test_treshold = int(len(df_sorted) * 0.98)
val_treshold = int(len(df_sorted) * 0.96)

train_val = df_sorted.head(test_treshold)
warm_test = df_sorted.tail(len(df_sorted) - test_treshold)
test = warm_test.loc[warm_test.groupby('user_id')['timestamp'].idxmax()]
warm_t = warm_test[~warm_test.index.isin(test.index)]


train = df_sorted.head(val_treshold)
test_val = df_sorted.tail(len(df_sorted) - val_treshold)
warm_val = test_val[~test_val.index.isin(warm_test.index)]
val = warm_val.loc[warm_val.groupby('user_id')['timestamp'].idxmax()]
warm_v = warm_val[~warm_val.index.isin(val.index)]




In [7]:
print("Train len: ", len(train))
print("Train users: ", len(train["user_id"].unique()))
print("Val len: ", len(val))
print("Val users: ", len(val["user_id"].unique()))
print("Test len: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Warm val len: ", len(warm_v))
print("Warm val users: ", len(warm_v["user_id"].unique()))
print("Warm test len: ", len(warm_t))
print("Warm test users: ", len(warm_t["user_id"].unique()))
print("test_val intersection users: ", np.intersect1d(val["user_id"].unique(), test["user_id"].unique()).shape[0])

Train len:  284483
Train users:  35226
Val len:  2645
Val users:  2645
Test len:  2368
Test users:  2368
Warm val len:  3282
Warm val users:  1229
Warm test len:  3559
Warm test users:  1160
test_val intersection users:  438


In [8]:
user_items = train.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Sports/train.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [9]:
train_warm_v = pd.concat([train, warm_v], ignore_index=True)
user_items = warm_v.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Sports/warm_val.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [10]:
filtered_train = train_warm_v[train_warm_v['user_id'].isin(val['user_id'])]
train_val = pd.concat([filtered_train, val], ignore_index=True)

user_items = train_val.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/Sports/val.txt', 'w') as f:
    for user_id, items in user_items.items():
        if len(items) > 1:
            f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [11]:
unique_indices = val["user_id"].unique()
with open('data2/Sports/val_users.txt', 'w') as f:
    for index in unique_indices:
        f.write(f"{index}\n")

# Чтение чисел из файла и сохранение их в список
with open('data2/Sports/val_users.txt', 'r') as f:
    index_list = [int(line.strip()) for line in f]

In [12]:
filtered_train = df_sorted[df_sorted['user_id'].isin(test['user_id'])]


user_items = filtered_train.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/Sports/test.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [13]:
user_items = df_sorted.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Sports/all_data.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")